In [36]:
import pandas as pd
import numpy as np

df = pd.read_csv(("../data/raw/pubmed_papers.csv"))
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embedding = model.encode(df['Abstract'].to_list(), show_progress_bar=True)
from sklearn.metrics.pairwise import cosine_similarity

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [37]:
def search_paper(query, top_k):
    embedding_query = model.encode(query)
    similarities = cosine_similarity([embedding_query], embedding)
    top_k_indx = np.argsort(similarities[0])[-top_k:][::-1]
    results= []
    
    for idx in top_k_indx:
        pmid = int(df.iloc[idx]['PMID'])
        title = df.iloc[idx]['Title']
        abstract = df.iloc[idx]['Abstract']
        sim = float(similarities[0][idx])

        results.append({
            'PMID':pmid,
            'Title':title,
            'Abstract':abstract,
            'Similarities':sim
            })
          
    return results

 
    

In [38]:
def build_context(results):
    context_parts = []
    for paper in results:
        title = paper['Title']
        abstract = paper['Abstract']

        formatted_text = f"""
            Paper

            Title:
            {title}

            Abstract:
            {abstract}

            --------------------------------------------------
            """

        context_parts.append(formatted_text)


    context = "\n".join(context_parts)
    return context




In [39]:
results = search_paper("lung cancer biomarkers", 1)
context = build_context(results)
print(context[:1000])




            Paper

            Title:
            Advances in early-stage lung cancer patients: from preanalytics to molecular analysis.

            Abstract:
            Lung Cancer (LC) remains one of the leading causes of cancer death worldwide. Molecular profiling of mandatory testing biomarkers significantly meliorates clinical outcome of advanced stage LC patients while intercepting early-stage lesions represents an unmet need. Recent advances in treatments for early-stage LC patients lay the basis for accelerating molecular analysis of actionable drivers before metastatic stages. In this scenario, liquid biopsy emerged as an innovative tool to guide clinical decision-making processes of early-stage LC patients. In addition, digital pathology is pioneering an integrative path to optimize morpho-molecular analysis. Undoubtedly, the lack of standardized preanalytical procedures significantly reduces the clinical availability of these approaches. This review aims to explore critic

In [40]:
print(len(context))

1207


LLM - Hugging Face

In [14]:
from transformers import pipeline
print(pipeline)


<function pipeline at 0x0000020892A5E140>


In [15]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

In [41]:
import transformers 
print(transformers.__version__)

5.10.2


In [42]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [43]:
type(model)

transformers.models.t5.modeling_t5.T5ForConditionalGeneration

In [49]:
query = "What biomarkers are useful for lung cancer diagnosis?"

prompt = f"""
You are a biomedical research assistant.

Read the scientific abstract below and answer the question in complete sentences.

Context:
{context}

Question:
What biomarkers are useful for lung cancer diagnosis?

Provide a summary of the biomarkers mentioned.
"""

In [50]:
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

In [51]:
outputs = model.generate(
    **inputs,
   max_new_tokens=100,
    num_beams=5,
    
)

In [52]:
answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

Molecular profiling of mandatory testing biomarkers significantly meliorates clinical outcome of advanced stage LC patients while intercepting early-stage lesions represents an unmet need. Recent advances in treatments for early-stage LC patients lay the basis for accelerating molecular analysis of actionable drivers before metastatic stages. In addition, digital pathology is pioneering an integrative path to optimize morpho-molecular analysis. Undoubtedly, the lack of standardized


In [53]:
print(inputs["input_ids"].shape)

torch.Size([1, 285])
